# ニューラルネットワーク

ニューラルネットワークは、入力に重みを掛けて足し合わせ、非線形関数を通し、出力と正解のずれが小さくなるように重みを更新するモデルです。最初は、1 個のニューロンが作る直線の境界から見ます。

OR は 1 本の直線で分けられますが、XOR は対角の 2 点を同じクラスにするため、直線だけでは分けられません。2 層 MLP は、隠れ層で入力を別の表現へ写してから判断します。逆伝播は、出力の誤差を各重みへ戻し、損失が下がる方向へ少しずつ更新する手続きです。

In [ ]:
import math
import random

random.seed(11)

X_or = [(0.0, 0.0), (0.0, 1.0), (1.0, 0.0), (1.0, 1.0)]
y_or = [0.0, 1.0, 1.0, 1.0]

X_xor = X_or[:]
y_xor = [0.0, 1.0, 1.0, 0.0]

print('OR:', list(zip(X_or, y_or)))
print('XOR:', list(zip(X_xor, y_xor)))

ロジスティック回帰は `p = sigmoid(w1*x1 + w2*x2 + b)` と書けます。`p` はクラス 1 である確率、`y` は正解ラベルです。BCE の勾配は `p - y` を入力へ掛ける形になり、予測が正解より大きすぎるなら重みを下げ、小さすぎるなら上げます。ここでは、重みが境界線の向きを決め、バイアスが境界線の位置を動かすことを確認します。

In [ ]:
def sigmoid(x):
    if x >= 0:
        e = math.exp(-x)
        return 1.0 / (1.0 + e)
    e = math.exp(x)
    return e / (1.0 + e)


def bce(p, y):
    p = min(1.0 - 1e-9, max(1e-9, p))
    return -(y * math.log(p) + (1.0 - y) * math.log(1.0 - p))


def train_logistic(X, y, steps=900, lr=0.45):
    w = [random.gauss(0.0, 0.2), random.gauss(0.0, 0.2)]
    b = 0.0
    history = []
    for step in range(steps):
        gw = [0.0, 0.0]
        gb = 0.0
        loss = 0.0
        for (x1, x2), target in zip(X, y):
            p = sigmoid(w[0] * x1 + w[1] * x2 + b)
            err = p - target
            gw[0] += err * x1 / len(X)
            gw[1] += err * x2 / len(X)
            gb += err / len(X)
            loss += bce(p, target) / len(X)
        w[0] -= lr * gw[0]
        w[1] -= lr * gw[1]
        b -= lr * gb
        if step % 180 == 0 or step == steps - 1:
            history.append((step, loss, w[:], b))
    return w, b, history

w_or, b_or, hist_or = train_logistic(X_or, y_or)
for step, loss, w, b in hist_or:
    print(step, 'loss=', round(loss, 3), 'w=', [round(v, 3) for v in w], 'b=', round(b, 3))

OR は単一ニューロンで解けます。次に同じモデルを XOR へ使うと、損失が下がりきらず、確率は 0.5 付近へ残ります。これは学習率や反復回数だけの問題ではありません。モデルの形が直線 1 本なので、対角の 2 群を分ける境界を作れないという表現力の限界です。

In [ ]:
def predict_logistic(X, w, b):
    return [sigmoid(w[0] * x1 + w[1] * x2 + b) for x1, x2 in X]

print('OR probs:', [round(p, 3) for p in predict_logistic(X_or, w_or, b_or)])
print('OR pred:', [int(p >= 0.5) for p in predict_logistic(X_or, w_or, b_or)])

w_xor, b_xor, hist_xor = train_logistic(X_xor, y_xor, steps=1300, lr=0.35)
print('XOR final loss:', round(hist_xor[-1][1], 3))
print('XOR probs:', [round(p, 3) for p in predict_logistic(X_xor, w_xor, b_xor)])
print('XOR pred:', [int(p >= 0.5) for p in predict_logistic(X_xor, w_xor, b_xor)])

2 層 MLP は `x -> tanh(W1 x + b1) -> sigmoid(W2 h + b2)` と進む。隠れ層があるため、XOR の対角パターンを別表現へ写せる。入力をそのまま直線で切るのではなく、隠れ層で複数の中間特徴を作ってから最後に判定する。

In [ ]:
def init_mlp(hidden=4):
    return {
        'W1': [[random.gauss(0.0, 0.7) for _ in range(hidden)] for _ in range(2)],
        'b1': [0.0 for _ in range(hidden)],
        'W2': [random.gauss(0.0, 0.7) for _ in range(hidden)],
        'b2': 0.0,
    }


def mlp_forward_one(x, params):
    h = []
    for j in range(len(params['b1'])):
        z = x[0] * params['W1'][0][j] + x[1] * params['W1'][1][j] + params['b1'][j]
        h.append(math.tanh(z))
    logit = sum(h[j] * params['W2'][j] for j in range(len(h))) + params['b2']
    return h, sigmoid(logit)


def mlp_loss(X, y, params):
    total = 0.0
    for x, target in zip(X, y):
        _, p = mlp_forward_one(x, params)
        total += bce(p, target)
    return total / len(X)

params0 = init_mlp()
print('initial MLP loss:', round(mlp_loss(X_xor, y_xor, params0), 3))

逆伝播では、出力誤差 `p-y` を `W2` へ戻し、さらに `tanh` の微分 `1-h^2` を通して `W1` へ戻す。後段の誤差が前段の重みに届くため、隠れ層はただの固定変換ではなく、最終損失に合わせて形を変える。

In [ ]:
def zero_grads(params):
    hidden = len(params['b1'])
    return {
        'W1': [[0.0 for _ in range(hidden)] for _ in range(2)],
        'b1': [0.0 for _ in range(hidden)],
        'W2': [0.0 for _ in range(hidden)],
        'b2': 0.0,
    }


def add_grads(grads, x, target, params, scale):
    h, p = mlp_forward_one(x, params)
    dlogit = (p - target) * scale
    for j in range(len(h)):
        grads['W2'][j] += h[j] * dlogit
    grads['b2'] += dlogit
    for j in range(len(h)):
        dh = params['W2'][j] * dlogit
        dz = dh * (1.0 - h[j] * h[j])
        grads['W1'][0][j] += x[0] * dz
        grads['W1'][1][j] += x[1] * dz
        grads['b1'][j] += dz


def apply_grads(params, grads, lr):
    for i in range(2):
        for j in range(len(params['b1'])):
            params['W1'][i][j] -= lr * grads['W1'][i][j]
    for j in range(len(params['b1'])):
        params['b1'][j] -= lr * grads['b1'][j]
        params['W2'][j] -= lr * grads['W2'][j]
    params['b2'] -= lr * grads['b2']


def train_mlp(X, y, steps=3500, lr=0.35, batch_size=4, seed=5):
    random.seed(seed)
    params = init_mlp()
    history = []
    for step in range(steps):
        if batch_size >= len(X):
            idxs = list(range(len(X)))
        else:
            idxs = [random.randrange(len(X)) for _ in range(batch_size)]
        scale = 1.0 / len(idxs)
        grads = zero_grads(params)
        for idx in idxs:
            add_grads(grads, X[idx], y[idx], params, scale=scale)
        apply_grads(params, grads, lr)
        if step % 700 == 0 or step == steps - 1:
            history.append((step, mlp_loss(X, y, params)))
    return params, history

params_xor, hist_mlp = train_mlp(X_xor, y_xor)
for step, loss in hist_mlp:
    print(step, 'loss=', round(loss, 4))

MLP は XOR を分けられる。単一ニューロンとの差は、入力を隠れ層で作り替えてから判断している点にある。

In [ ]:
def mlp_probs(X, params):
    return [mlp_forward_one(x, params)[1] for x in X]

probs = mlp_probs(X_xor, params_xor)
print('MLP probs:', [round(p, 3) for p in probs])
print('MLP pred:', [int(p >= 0.5) for p in probs])
print('target:', [int(v) for v in y_xor])

勾配チェックは、逆伝播で計算した勾配が実装ミスを含んでいないかを調べる検査です。1 つのパラメータをほんの少し増減させ、損失がどれだけ変わるかを数値微分で測り、逆伝播の値と比べます。

In [ ]:
def one_batch_grads(X, y, params):
    grads = zero_grads(params)
    for x, target in zip(X, y):
        add_grads(grads, x, target, params, scale=1.0 / len(X))
    return grads


def numeric_grad_w10(params, eps=1e-5):
    params['W1'][1][0] += eps
    plus = mlp_loss(X_xor, y_xor, params)
    params['W1'][1][0] -= 2.0 * eps
    minus = mlp_loss(X_xor, y_xor, params)
    params['W1'][1][0] += eps
    return (plus - minus) / (2.0 * eps)

grads = one_batch_grads(X_xor, y_xor, params_xor)
print('backprop grad:', round(grads['W1'][1][0], 6))
print('numeric grad:', round(numeric_grad_w10(params_xor), 6))
print('abs diff:', round(abs(grads['W1'][1][0] - numeric_grad_w10(params_xor)), 8))

ミニバッチでは一部のデータだけで更新するため、損失は揺れやすい。全件更新は滑らかだが、データが大きいと 1 回の更新が重くなる。

In [ ]:
_, hist_full = train_mlp(X_xor, y_xor, steps=2200, lr=0.35, batch_size=4, seed=8)
_, hist_mini = train_mlp(X_xor, y_xor, steps=2200, lr=0.35, batch_size=2, seed=8)

print('full batch history:', [(s, round(v, 4)) for s, v in hist_full])
print('mini batch history:', [(s, round(v, 4)) for s, v in hist_mini])

単一ニューロンは直線で分けるモデルであり、XOR のような対角パターンでは限界が出る。隠れ層は入力表現を作り替え、逆伝播はその各部品へ誤差を戻す。深いネットワークも、この順伝播、損失、逆伝播、更新の繰り返しで学習する。